# Module 7 · — CI/CD & Automation: Pipeline Orchestration & Behavioral Gating

**Architectural Layer:** CI/CD & Automation  
**Toolchain:** Kubeflow Pipelines (KFP) · GitHub Actions · MLflow  
**Objective:** Build automated ML pipelines with containerized components, behavioral test gates, and programmatic DAG compilation.

---

## 🧠 What is CI/CD for Machine Learning and Why Is It Different?

### CI/CD in Traditional Software
- **CI (Continuous Integration):** Every time a developer pushes code, automated tests run. If tests pass, the code is merged.
- **CD (Continuous Delivery):** After merging, the software is automatically deployed to production.

### 🤔 Why Can't We Just Use Normal CI/CD for ML?

Because ML has a fundamental difference: **the output is non-deterministic.**

| Traditional Software CI/CD | ML CI/CD |
|---------------------------|----------|
| Tests are binary: pass/fail | Tests are statistical: "is accuracy > 80%?" |
| Code logic is the product | Code + Data + Model weights are the product |
| Bugs are in the code | Bugs can be in code, data, OR learned behavior |
| Output is deterministic | Output varies with random seeds, data order |
| Deploy once, runs forever | Model degrades over time (data drift) |

### Real-World Scenario

Your data science team trains a new sentiment analysis model. Traditional CI/CD would check: "Does the code compile? Do unit tests pass?" But for ML, you also need:
- "Does the model accuracy exceed 80% on the test set?" (performance gate)
- "Does replacing 'John' with 'Jane' change the sentiment prediction?" (fairness/invariance gate)
- "Is the model robust to typos?" (robustness gate)

If ANY gate fails, the model MUST NOT reach production.

### 📚 What is a Pipeline DAG?

**DAG** = Directed Acyclic Graph. It's a flowchart that shows which steps must happen in which order.

```
Preprocess Data → Train Model → Evaluate → [Quality Gate] → Register
```

"Directed" means steps flow in one direction. "Acyclic" means no loops.

**🤔 Why not just run steps one after another in a script?**

| Linear Script | Pipeline DAG |
|--------------|-------------|
| Steps 3 and 4 run sequentially even if independent | Independent steps run in parallel |
| If step 3 fails, you restart from step 1 | If step 3 fails, only rerun step 3 |
| No visibility into which step is running | Dashboard shows progress per step |
| Hard to reuse individual steps | Each step is a reusable component |

### ⚖️ Trade-offs: KFP vs Airflow vs GitHub Actions

| Tool | Best For | Limitations |
|------|----------|-------------|
| **Kubeflow Pipelines** | ML-specific pipelines on Kubernetes | Requires K8s cluster, steep learning curve |
| **Apache Airflow** | General data pipelines, scheduling | Not ML-specific, no GPU support out of the box |
| **GitHub Actions** | Code CI/CD, triggering pipelines | Time limits, not designed for heavy compute |

**In practice, you combine them:** GitHub Actions triggers the pipeline, KFP orchestrates the ML steps, Airflow handles data scheduling.

---

## 📑 Table of Contents

* [🧠 What is CI/CD for Machine Learning and Why Is It Different?](#what-is-cicd-for-machine-learning-and-why-is-it-different)
  * [CI/CD in Traditional Software](#cicd-in-traditional-software)
  * [🤔 Why Can't We Just Use Normal CI/CD for ML?](#why-cant-we-just-use-normal-cicd-for-ml)
  * [Real-World Scenario](#real-world-scenario)
  * [📚 What is a Pipeline DAG?](#what-is-a-pipeline-dag)
  * [⚖️ Trade-offs: KFP vs Airflow vs GitHub Actions](#trade-offs-kfp-vs-airflow-vs-github-actions)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Pipeline Component Design](#1-pipeline-component-design)
  * [🤔 What is a Pipeline Component?](#what-is-a-pipeline-component)
* [2 · Behavioral Testing Gate](#2-behavioral-testing-gate)
  * [🤔 What is a "Quality Gate"?](#what-is-a-quality-gate)
  * [Types of ML Tests](#types-of-ml-tests)
  * [Real-World Scenario: Why Invariance Tests Matter](#real-world-scenario-why-invariance-tests-matter)
* [3 · DAG Composition & Compilation](#3-dag-composition-compilation)
  * [🤔 How Do Steps Connect?](#how-do-steps-connect)
  * [🤔 Why Parallel Execution Matters](#why-parallel-execution-matters)
* [4 · Compilation & Trigger Simulation](#4-compilation-trigger-simulation)
  * [🤔 What Does "Compiling" a Pipeline Mean?](#what-does-compiling-a-pipeline-mean)
  * [🤔 How Does GitHub Actions Trigger This?](#how-does-github-actions-trigger-this)
* [5 · Automated Validation with deepchecks](#5-automated-validation-with-deepchecks)
  * [🤔 What is deepchecks?](#what-is-deepchecks)
* [6 · Workflow Orchestration with Metaflow](#6-workflow-orchestration-with-metaflow)
  * [🤔 Why Metaflow?](#why-metaflow)
* [7 · GitHub Actions CI/CD for ML](#7-github-actions-cicd-for-ml)
  * [🤔 How Does GitHub Actions Fit into ML?](#how-does-github-actions-fit-into-ml)
* [Knowledge Check](#knowledge-check)
  * [Question 1: Quality Gates](#question-1-quality-gates)
  * [Question 2: KFP Components](#question-2-kfp-components)
  * [Question 3: Invariance Testing](#question-3-invariance-testing)
  * [Question 4: DAG vs Linear Script](#question-4-dag-vs-linear-script)
* [Practical Practice](#practical-practice)
  * [Exercise 1: Design Quality Gates](#exercise-1-design-quality-gates)
  * [Exercise 2: Write a KFP Component](#exercise-2-write-a-kfp-component)
  * [Exercise 3: GitHub Actions Trigger](#exercise-3-github-actions-trigger)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 Key Questions](#key-questions)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** MLOps is DevOps applied to ML. If deploying an updated model requires manual steps on a laptop, you do not have an ML system; you have a science project. Seniors automate testing and deployment in pipelines.

**Mental Model:** A CI/CD pipeline is an automated assembly line. Code/models are pushed to Git, tested by a robotic inspector (CI), and safely deployed to the storefront (CD) without human intervention.

**Common Junior Pitfall:** Only triggering pipelines when *code* changes. In ML, models must be retrained and redeployed automatically when *data* distributions change, requiring Data-Triggered ML pipelines.

---


## 1 · Pipeline Component Design

### 🤔 What is a Pipeline Component?

A **component** is a self-contained, reusable step in your pipeline. Think of it like a LEGO brick — each brick does one thing, and you snap them together to build complex structures.

Each component:
- Has defined **inputs** and **outputs**
- Runs in its own **container** (isolated environment with its own dependencies)
- Can be **reused** across different pipelines
- Is **independently testable**

**🤔 Why containers for each component?**  
Because different steps need different dependencies. Your preprocessing step needs pandas but not PyTorch. Your training step needs PyTorch but not pandas. Containers keep them isolated so they don't conflict.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q kfp==2.7.0 mlflow pandas numpy scikit-learn requests pyyaml

In [ ]:
# Cell 2 — Define data preprocessing component
#
# WHAT IS @dsl.component?
# It's a Python decorator that turns a regular function into a KFP component.
# KFP will:
#   1. Package this function into a container image
#   2. Install the specified packages
#   3. Run it as an isolated step in the pipeline
#
# IMPORTANT: The function must be SELF-CONTAINED.
# It cannot import modules defined outside the function.
# All imports must be INSIDE the function body.
# This is because the function runs in a separate container!
#
# Input/Output types:
# - Input[Dataset]: receives a dataset from a previous step
# - Output[Dataset]: produces a dataset for the next step
# - Output[Metrics]: logs metrics visible in the KFP dashboard

from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, Artifact

@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["pandas", "numpy", "scikit-learn"],
)
def preprocess_data(
    raw_data_path: str,
    test_split: float,
    train_data: Output[Dataset],
    test_data: Output[Dataset],
    preprocessing_metrics: Output[Metrics],
):
    """Ingest, validate, and split raw data for training."""
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split

    # Simulate loading data (in production: read from S3, BigQuery, etc.)
    np.random.seed(42)
    n_samples = 10000
    data = pd.DataFrame({
        "feature_1": np.random.randn(n_samples),
        "feature_2": np.random.exponential(2, n_samples),
        "feature_3": np.random.choice(["A", "B", "C"], n_samples),
        "target": np.random.binomial(1, 0.3, n_samples),
    })

    # Validate
    assert data.isnull().sum().sum() == 0, "Data contains nulls!"
    data = pd.get_dummies(data, columns=["feature_3"], drop_first=True)

    # Split with stratification (preserve class balance)
    train_df, test_df = train_test_split(data, test_size=test_split, random_state=42, stratify=data["target"])
    train_df.to_parquet(train_data.path)
    test_df.to_parquet(test_data.path)

    # Log metrics (visible in KFP dashboard)
    preprocessing_metrics.log_metric("total_samples", n_samples)
    preprocessing_metrics.log_metric("train_samples", len(train_df))
    preprocessing_metrics.log_metric("test_samples", len(test_df))

print("✅ Preprocessing component defined")

In [ ]:
# Cell 3 — Define training component
#
# This component receives the preprocessed data and trains a model.
# It saves the trained model as a serialized artifact.
#
# 🤔 Why pickle for model serialization?
# Pickle is Python's built-in serialization. For scikit-learn models,
# it's the simplest option. For deep learning models, you'd use
# model.save_pretrained() (HuggingFace) or torch.save() (PyTorch).

@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["pandas", "numpy", "scikit-learn", "mlflow"],
)
def train_model(
    train_data: Input[Dataset],
    learning_rate: float,
    n_estimators: int,
    model_artifact: Output[Model],
    training_metrics: Output[Metrics],
):
    """Train a gradient boosted classifier."""
    import pandas as pd
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    import pickle

    df = pd.read_parquet(train_data.path)
    X, y = df.drop("target", axis=1), df["target"]

    clf = GradientBoostingClassifier(learning_rate=learning_rate, n_estimators=n_estimators, max_depth=5, random_state=42)
    clf.fit(X, y)

    y_pred = clf.predict(X)
    y_proba = clf.predict_proba(X)[:, 1]

    training_metrics.log_metric("train_accuracy", accuracy_score(y, y_pred))
    training_metrics.log_metric("train_f1", f1_score(y, y_pred))
    training_metrics.log_metric("train_auc", roc_auc_score(y, y_proba))

    with open(model_artifact.path, "wb") as f:
        pickle.dump(clf, f)

print("✅ Training component defined")

---

## 2 · Behavioral Testing Gate

### 🤔 What is a "Quality Gate"?

A quality gate is like a bouncer at a club. The model MUST meet certain minimum requirements to proceed. If it doesn't, the pipeline STOPS and the model is not deployed.

### Types of ML Tests

| Test Type | What It Checks | Example | When It Fails |
|-----------|---------------|---------|---------------|
| **Performance gate** | Minimum accuracy/F1 | "Accuracy must be > 80%" | Model isn't good enough |
| **Invariance test** | Predictions stable to irrelevant changes | "Changing a name shouldn't change sentiment" | Model learned biased shortcuts |
| **Directional test** | Predictions change correctly for relevant changes | "Adding 'not' should flip sentiment" | Model ignores important information |
| **Sanity check** | Model is better than random | "AUC must be > 0.5" | Model is fundamentally broken |

### Real-World Scenario: Why Invariance Tests Matter

A resume screening model was deployed at a major company. It scored resumes to determine interview candidates. After investigation, they found:
- Changing "John" to "Emily" lowered the score
- Changing "Stanford" to "community college" dramatically lowered it

An invariance test would have caught this BEFORE deployment: "Changing a person's name should NOT change the hiring score."

In [ ]:
# Cell 4 — Define behavioral evaluation component
#
# This is the QUALITY GATE. It checks the model against minimum thresholds.
# If any threshold is not met, the component RAISES AN ERROR,
# which causes the pipeline to STOP.
#
# 🤔 Why raise an error instead of just logging a warning?
# Because a warning gets ignored. An error BLOCKS the pipeline.
# In production, blocking is exactly what you want.
# A bad model in production is worse than no new model at all.

@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["pandas", "numpy", "scikit-learn"],
)
def evaluate_model(
    model_artifact: Input[Model],
    test_data: Input[Dataset],
    min_accuracy: float,     # minimum acceptable accuracy
    min_f1: float,           # minimum acceptable F1 score
    evaluation_metrics: Output[Metrics],
    gate_result: Output[Artifact],
):
    """Evaluate model against minimum functional thresholds."""
    import pandas as pd
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    import pickle, json

    with open(model_artifact.path, "rb") as f:
        clf = pickle.load(f)
    df = pd.read_parquet(test_data.path)
    X, y = df.drop("target", axis=1), df["target"]

    y_pred = clf.predict(X)
    y_proba = clf.predict_proba(X)[:, 1]
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    auc = roc_auc_score(y, y_proba)

    evaluation_metrics.log_metric("test_accuracy", acc)
    evaluation_metrics.log_metric("test_f1", f1)
    evaluation_metrics.log_metric("test_auc", auc)

    # THE QUALITY GATE
    gates = {
        "accuracy_gate": acc >= min_accuracy,
        "f1_gate": f1 >= min_f1,
        "auc_sanity": auc > 0.5,
    }
    passed = all(gates.values())

    with open(gate_result.path, "w") as f:
        json.dump({"passed": passed, "gates": gates, "metrics": {"accuracy": acc, "f1": f1, "auc": auc}}, f, indent=2)

    if not passed:
        raise ValueError(f"Quality gate FAILED: {gates}")

print("✅ Evaluation component with behavioral gates defined")

In [ ]:
# Cell 5 — Define NLP invariance testing component
#
# WHAT IS AN INVARIANCE TEST?
# A test that checks: "If I make a change that SHOULDN'T affect the output,
# does the output actually stay the same?"
#
# Examples:
# - Replacing "product" with "item" → sentiment should stay the same
# - Replacing "Alex" with "Jordan" → prediction should stay the same
# - Adding "not" → sentiment SHOULD change (this is a directional test)
#
# 🤔 What if the model fails invariance tests?
# It means the model learned spurious correlations (shortcuts).
# For example, it might associate certain names with certain outcomes.
# This is both a quality AND fairness issue.

@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["numpy"],
)
def nlp_invariance_test(invariance_result: Output[Artifact]):
    """Test NLP model invariance: identity substitutions must not change predictions."""
    import json

    test_cases = [
        {"type": "identity_substitution", "original": "The product is excellent.", "modified": "The item is excellent.",
         "expected": "same_sentiment", "original_score": 0.92, "modified_score": 0.89},
        {"type": "negation_check", "original": "The service was great.", "modified": "The service was not great.",
         "expected": "different_sentiment", "original_score": 0.95, "modified_score": 0.15},
        {"type": "name_independence", "original": "Alex submitted the application.", "modified": "Jordan submitted the application.",
         "expected": "same_prediction", "original_score": 0.88, "modified_score": 0.87},
    ]

    results = []
    for tc in test_cases:
        diff = abs(tc["original_score"] - tc["modified_score"])
        passed = diff < 0.1 if "same" in tc["expected"] else diff > 0.3
        results.append({**tc, "passed": passed, "score_diff": diff})

    all_passed = all(r["passed"] for r in results)
    with open(invariance_result.path, "w") as f:
        json.dump({"all_passed": all_passed, "tests": results}, f, indent=2)

    if not all_passed:
        raise ValueError(f"Invariance tests FAILED: {len([r for r in results if not r['passed']])} failures")

print("✅ NLP invariance test component defined")

In [ ]:
# Cell 6 — Define model registration component
#
# This step ONLY runs if both the quality gate AND invariance tests pass.
# It registers the model in MLflow, making it available for deployment.
#
# 🤔 Why check gate_result AGAIN here?
# Defense in depth. Even though KFP would stop the pipeline if
# the evaluation step failed, we double-check here.
# In critical systems, redundant checks prevent mistakes.

@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["mlflow"],
)
def register_model(
    model_artifact: Input[Model],
    gate_result: Input[Artifact],
    model_name: str,
    registry_uri: str,
):
    """Register model in MLflow registry only if gates passed."""
    import json, mlflow

    with open(gate_result.path) as f:
        gate_data = json.load(f)
    if not gate_data["passed"]:
        raise RuntimeError("Cannot register: quality gates did not pass")

    mlflow.set_tracking_uri(registry_uri)
    with mlflow.start_run(run_name=f"pipeline-{model_name}"):
        mlflow.log_artifact(model_artifact.path, "model")
        mlflow.log_metrics(gate_data["metrics"])
        result = mlflow.register_model(f"runs:/{mlflow.active_run().info.run_id}/model", model_name)
    print(f"Model registered: {model_name} v{result.version}")

print("✅ Registration component defined")

---

## 3 · DAG Composition & Compilation

### 🤔 How Do Steps Connect?

Each step declares its inputs and outputs. KFP automatically chains them:
- `preprocess_data` outputs `train_data` and `test_data`
- `train_model` takes `train_data` as input → it MUST wait for preprocess to finish
- `evaluate_model` takes both `model_artifact` and `test_data` → it waits for BOTH
- Steps without data dependencies can run in PARALLEL

### 🤔 Why Parallel Execution Matters

If your evaluation takes 30 minutes and invariance testing takes 15 minutes, running them sequentially takes 45 minutes. Running them in parallel takes only 30 minutes. At scale, this saves hours per day.

In [ ]:
# Cell 7 — Compose the full pipeline DAG
#
# This function DEFINES the pipeline structure.
# It does NOT run the pipeline — it describes WHAT should happen.
#
# The @dsl.pipeline decorator marks this as a KFP pipeline.
# Inside, we call our components and wire their inputs/outputs together.
#
# KEY: register_task.after(invariance_task)
# This creates an EXPLICIT dependency: registration must wait for
# invariance tests to complete, even though there's no data flowing
# between them. This is a "control dependency" vs a "data dependency."

@dsl.pipeline(
    name="mlops-training-pipeline",
    description="End-to-end ML pipeline: preprocess → train → evaluate → gate → register",
)
def training_pipeline(
    raw_data_path: str = "s3://mlops-data/raw/",
    test_split: float = 0.2,
    learning_rate: float = 0.1,
    n_estimators: int = 100,
    min_accuracy: float = 0.65,
    min_f1: float = 0.5,
    model_name: str = "gradient-boost-classifier",
    registry_uri: str = "sqlite:///mlflow.db",
):
    # Step 1: Preprocess (runs first, no dependencies)
    preprocess_task = preprocess_data(raw_data_path=raw_data_path, test_split=test_split)

    # Step 2: Train (depends on preprocess → data dependency)
    train_task = train_model(train_data=preprocess_task.outputs["train_data"], learning_rate=learning_rate, n_estimators=n_estimators)

    # Step 3: Evaluate with quality gates (depends on train + preprocess)
    eval_task = evaluate_model(model_artifact=train_task.outputs["model_artifact"], test_data=preprocess_task.outputs["test_data"], min_accuracy=min_accuracy, min_f1=min_f1)

    # Step 4: NLP invariance tests (runs in PARALLEL with eval!)
    invariance_task = nlp_invariance_test()

    # Step 5: Register only if BOTH gates pass
    register_task = register_model(model_artifact=train_task.outputs["model_artifact"], gate_result=eval_task.outputs["gate_result"], model_name=model_name, registry_uri=registry_uri)
    register_task.after(invariance_task)  # control dependency: wait for invariance

print("✅ Pipeline DAG composed")
print("   Flow: preprocess → train → [evaluate | invariance_test] → register")

In [ ]:
# Cell 8 — Visualize the DAG structure
dag_structure = """
Pipeline: mlops-training-pipeline
══════════════════════════════════

  ┌─────────────────┐
  │  preprocess_data │   ← Step 1: Ingest & validate data
  └────────┬────────┘
           │
  ┌────────▼────────┐
  │   train_model   │    ← Step 2: Train on preprocessed data
  └────────┬────────┘
           │
    ┌──────┴──────┐       ← Steps 3 & 4 run in PARALLEL
    ▼             ▼
┌──────────┐ ┌──────────────┐
│ evaluate │ │ invariance   │
│ (gates)  │ │ (robustness) │
└────┬─────┘ └──────┬───────┘
     │              │
     └──────┬───────┘  ← Both must pass
            ▼
    ┌───────────────┐
    │register_model │  ← Step 5: Only if ALL gates pass
    └───────────────┘
"""
print(dag_structure)

In [ ]:
# Cell 9 — Display pipeline parameters and defaults
import inspect

sig = inspect.signature(training_pipeline)
print("📋 Pipeline Parameters:")
print(f"{'Parameter':<25} {'Default':<30} {'Type'}")
print("─" * 70)
for name, param in sig.parameters.items():
    default = param.default if param.default != inspect.Parameter.empty else "<required>"
    annotation = param.annotation.__name__ if hasattr(param.annotation, '__name__') else str(param.annotation)
    print(f"{name:<25} {str(default):<30} {annotation}")

---

## 4 · Compilation & Trigger Simulation

### 🤔 What Does "Compiling" a Pipeline Mean?

Compilation converts your Python pipeline definition into a YAML file that Kubernetes can execute. Think of it as:
- **Python code** = blueprint written in a language humans understand
- **YAML file** = translated blueprint that the Kubernetes cluster understands

### 🤔 How Does GitHub Actions Trigger This?

```
Developer pushes code → GitHub Actions webhook fires → 
Pipeline compiled → Submitted to Kubernetes → 
Steps execute → If gates pass → Model registered
```

**The entire flow is automatic.** No human intervention needed. This is "Continuous Training" — every time data or code changes, the model is retrained and validated automatically.

In [ ]:
# Cell 10 — Compile pipeline to YAML
from kfp import compiler
import os

PIPELINE_FILE = "training_pipeline.yaml"
compiler.Compiler().compile(pipeline_func=training_pipeline, package_path=PIPELINE_FILE)

file_size = os.path.getsize(PIPELINE_FILE)
print(f"✅ Pipeline compiled to: {PIPELINE_FILE} ({file_size/1024:.1f} KB)")

with open(PIPELINE_FILE) as f:
    lines = f.readlines()[:30]
print(f"\n📄 First 30 lines of compiled spec:")
print("".join(lines))

In [ ]:
# Cell 11 — Simulate GitHub Actions webhook trigger
#
# In production, this payload would be sent by GitHub when someone
# pushes to the main branch. The CI/CD system receives it and
# submits the pipeline to the Kubernetes cluster.

import json
from datetime import datetime

webhook_payload = {
    "event": "push",
    "ref": "refs/heads/main",
    "repository": "org/mlops-training-pipeline",
    "commit": {
        "sha": "a1b2c3d4e5f6789012345678",
        "message": "feat: update training data v2.3",
        "author": "data-engineer@company.com",
        "timestamp": datetime.utcnow().isoformat() + "Z",
    },
    "pipeline_trigger": {
        "pipeline_name": "mlops-training-pipeline",
        "parameters": {"raw_data_path": "s3://mlops-data/raw/v2.3/", "learning_rate": 0.05, "n_estimators": 200},
    },
}

print("🔔 GitHub Actions Webhook Simulation:")
print(json.dumps(webhook_payload, indent=2))

In [ ]:
# Cell 12 — Simulate pipeline run tracking
import uuid

run_id = str(uuid.uuid4())[:8]
run_log = {
    "run_id": run_id,
    "status": "Succeeded",
    "steps": [
        {"name": "preprocess_data", "status": "✅", "duration": "45s", "output": "8000 train / 2000 test"},
        {"name": "train_model", "status": "✅", "duration": "2m 15s", "output": "train_acc=0.87"},
        {"name": "evaluate_model", "status": "✅", "duration": "30s", "output": "test_acc=0.82, f1=0.71"},
        {"name": "nlp_invariance_test", "status": "✅", "duration": "15s", "output": "3/3 passed"},
        {"name": "register_model", "status": "✅", "duration": "10s", "output": "v3 registered"},
    ],
    "total_duration": "3m 55s",
}

print(f"🚀 Pipeline Run: {run_id}")
for step in run_log["steps"]:
    print(f"   {step['status']} {step['name']} ({step['duration']}) → {step['output']}")
print(f"\n✅ Pipeline completed — model v3 ready for deployment")

---

## 5 · Automated Validation with deepchecks

### 🤔 What is deepchecks?

deepchecks is a testing library that automatically checks your data AND model for common problems:
- Data integrity (missing values, duplicates, outliers)
- Model evaluation (performance degradation, overfitting)
- Train-test validation (data leakage, distribution mismatch)

**🤔 When to use deepchecks vs custom tests?**
- **deepchecks**: Quick, comprehensive baseline checks (80% of what you need)
- **Custom tests**: Domain-specific requirements (e.g., fairness, latency SLAs)

In [ ]:
# Cell — deepchecks data integrity and model evaluation suites
!pip install -q deepchecks
from deepchecks.tabular import Dataset
from deepchecks.tabular.suites import data_integrity, model_evaluation
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import pandas as pd

X, y = make_classification(n_samples=2000, n_features=10, random_state=42)
df = pd.DataFrame(X, columns=[f'feat_{i}' for i in range(10)])
df['target'] = y
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_ds = Dataset(train_df, label='target', cat_features=[])
test_ds = Dataset(test_df, label='target', cat_features=[])

# Data integrity checks
integrity = data_integrity()
integrity_result = integrity.run(train_ds)
print("📊 Data Integrity Results:")
integrity_result.show()

# Model evaluation checks
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(train_df.drop('target', axis=1), train_df['target'])
eval_suite = model_evaluation()
eval_result = eval_suite.run(train_ds, test_ds, clf)
print("📊 Model Evaluation Results:")
eval_result.show()
# eval_result.save_as_html('deepchecks_report.html')  # CI would review this

---

## 6 · Workflow Orchestration with Metaflow

### 🤔 Why Metaflow?

Metaflow was built by Netflix for data scientists who want to write production workflows **in pure Python** — no YAML, no K8s knowledge needed.

| Feature | Metaflow | Airflow | Kubeflow |
|---------|---------|---------|----------|
| Language | Pure Python | Python + YAML | Python + K8s |
| Learning curve | Easy | Moderate | Steep |
| Local → Cloud | One-line change | Major refactor | Already cloud |
| Best for | Data scientists | Data engineers | ML platform teams |

In [ ]:
# Cell — Metaflow workflow definition
# Metaflow uses @step decorators on methods of a FlowSpec class.
# Each step runs in its own isolated environment.
# State (self.xxx) is automatically serialized between steps.

# NOTE: This defines the flow structure. To run: python flow.py run

flow_code = '''
from metaflow import FlowSpec, step

class TrainingPipeline(FlowSpec):
    @step
    def start(self):
        """Initialize pipeline parameters."""
        self.experiment_name = 'loan-approval-v3'
        self.test_size = 0.2
        self.next(self.load_data)

    @step
    def load_data(self):
        """Load and validate data from feature store."""
        from sklearn.datasets import make_classification
        self.X, self.y = make_classification(n_samples=5000, n_features=20)
        print(f'Loaded {len(self.y)} samples')
        self.next(self.train_model)

    @step
    def train_model(self):
        """Train with tracked hyperparameters."""
        from sklearn.ensemble import GradientBoostingClassifier
        from sklearn.model_selection import train_test_split
        X_tr, X_te, y_tr, y_te = train_test_split(self.X, self.y, test_size=self.test_size)
        self.model = GradientBoostingClassifier(n_estimators=200, max_depth=5)
        self.model.fit(X_tr, y_tr)
        self.accuracy = self.model.score(X_te, y_te)
        self.next(self.evaluate)

    @step
    def evaluate(self):
        """Gate: block deployment if accuracy < threshold."""
        self.deploy_approved = self.accuracy >= 0.85
        print(f'Accuracy: {self.accuracy:.4f} -> {"APPROVED" if self.deploy_approved else "BLOCKED"}')
        self.next(self.end)

    @step
    def end(self):
        """Pipeline complete."""
        print(f'Pipeline finished. Deploy: {self.deploy_approved}')

if __name__ == "__main__":
    TrainingPipeline()
'''

with open('metaflow_pipeline.py', 'w') as f:
    f.write(flow_code)
print("📄 Metaflow pipeline written to metaflow_pipeline.py")
print("   Run with: python metaflow_pipeline.py run")
print(flow_code)

---

## 7 · GitHub Actions CI/CD for ML

### 🤔 How Does GitHub Actions Fit into ML?

GitHub Actions triggers automated workflows on events (push, PR, schedule). For ML:

```
Push to main → Run tests → Validate data → Train model → Build Docker → Deploy
```

The YAML below defines this entire pipeline as code.

In [ ]:
# Cell — Generate GitHub Actions workflow YAML

gh_actions_yaml = '''
name: ML Pipeline CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  validate-and-deploy:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run deepchecks data validation
        run: python -m pytest tests/test_data_integrity.py -v

      - name: Run model evaluation tests
        run: python -m pytest tests/test_model_quality.py -v

      - name: Run behavioral/invariance tests
        run: python -m pytest tests/test_behavioral.py -v

      - name: Build Docker image (only if all tests pass)
        if: success() && github.ref == 'refs/heads/main'
        run: |
          docker build -t registry.company.com/ml-service:${{ github.sha }} .
          docker push registry.company.com/ml-service:${{ github.sha }}

      - name: Deploy to staging
        if: success() && github.ref == 'refs/heads/main'
        run: kubectl set image deployment/ml-service ml-service=registry.company.com/ml-service:${{ github.sha }}
'''

with open('github_actions_ml.yaml', 'w') as f:
    f.write(gh_actions_yaml)
print("📄 GitHub Actions Workflow:")
print(gh_actions_yaml)

---
## Knowledge Check

### Question 1: Quality Gates
Your model achieves 82% accuracy (threshold: 80%) but F1 = 0.45 (threshold: 0.50). Should the pipeline deploy the model?

<details>
<summary>Click to reveal answer</summary>

**No.** Quality gates use AND logic -- ALL gates must pass. Even though accuracy passes (82% > 80%), the F1 gate fails (0.45 < 0.50). The model is blocked from deployment. This is intentional: accuracy alone can be misleading on imbalanced datasets (as we learned in NB13).
</details>

### Question 2: KFP Components
Why must all imports be INSIDE the `@dsl.component` function body, not at the top of the file?

<details>
<summary>Click to reveal answer</summary>

Each KFP component runs in its own **Docker container**. The container only has the packages specified in `packages_to_install`. Imports at the top of your notebook/file are NOT available inside the container. The function body is serialized and sent to the container, so all imports must be inline.
</details>

### Question 3: Invariance Testing
A sentiment model gives score 0.92 for 'The product is excellent' but 0.45 for 'The item is excellent'. Is this a problem?

<details>
<summary>Click to reveal answer</summary>

**Yes!** 'product' and 'item' are synonyms -- this substitution should NOT change the sentiment score significantly. The 0.47 difference (0.92 vs 0.45) indicates the model learned a **spurious correlation** with the word 'product'. The invariance test would flag this: score_diff = 0.47 > 0.1 threshold = FAIL.
</details>

### Question 4: DAG vs Linear Script
Your pipeline has 5 steps. Step 3 fails. What happens in a DAG-based pipeline vs a linear script?

<details>
<summary>Click to reveal answer</summary>

**DAG pipeline:** Only step 3 fails. Steps 1 and 2 are cached/completed. You fix the bug and rerun from step 3 only (saving the time of steps 1-2). Steps 4-5 are skipped until step 3 succeeds. **Linear script:** The entire script crashes at step 3. You must restart from step 1 because there's no caching. All previous work is lost.
</details>


---
## Practical Practice

### Exercise 1: Design Quality Gates
You're deploying a medical diagnosis model. Design a set of quality gates (at least 4) with specific thresholds and explain why each matters.

### Exercise 2: Write a KFP Component
Write a `@dsl.component` that takes a model path and a test dataset path, loads the model, runs predictions, and outputs accuracy and F1 as metrics.

### Exercise 3: GitHub Actions Trigger
Write a GitHub Actions step that only runs model deployment when: (a) the branch is `main`, (b) all tests passed, and (c) the commit message contains `[deploy]`.


In [ ]:
# ===== EXERCISE SOLUTIONS =====

print('Ex 1: Medical Diagnosis Quality Gates')
gates = {
    'recall_gate': ('Recall >= 0.95', 'Missing a disease (false negative) could be fatal'),
    'precision_gate': ('Precision >= 0.70', 'Too many false positives waste resources'),
    'auc_gate': ('AUC >= 0.90', 'Model must separate classes well overall'),
    'fairness_gate': ('Max demographic disparity < 5%', 'Model must perform equally across demographics'),
    'calibration_gate': ('Brier score < 0.15', 'Predicted probabilities should match actual frequencies'),
}
for gate, (threshold, reason) in gates.items():
    print(f'  {gate}: {threshold}')
    print(f'    Why: {reason}')

print('\nEx 2: KFP Evaluation Component')
print('''@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["pandas", "scikit-learn"],
)
def evaluate(model_path: Input[Model], test_data: Input[Dataset],
             metrics: Output[Metrics]):
    import pandas as pd
    from sklearn.metrics import accuracy_score, f1_score
    import pickle
    with open(model_path.path, "rb") as f:
        model = pickle.load(f)
    df = pd.read_parquet(test_data.path)
    X, y = df.drop("target", axis=1), df["target"]
    y_pred = model.predict(X)
    metrics.log_metric("accuracy", accuracy_score(y, y_pred))
    metrics.log_metric("f1", f1_score(y, y_pred))''')

print('\nEx 3: Conditional GitHub Actions Deploy')
print('''- name: Deploy model
  if: >
    success() &&
    github.ref == "refs/heads/main" &&
    contains(github.event.head_commit.message, "[deploy]")
  run: kubectl apply -f deployment.yaml''')


---

## 🎯 Summary & Key Takeaways

| Step | Action | Why |
|------|--------|-----|
| 1 | Containerized components | Isolation, reusability, independent dependencies |
| 2 | Behavioral quality gates | Block bad models from reaching production |
| 3 | DAG composition | Parallel execution, clear dependencies |
| 4 | YAML compilation + webhook trigger | Fully automated continuous training |

### 🧠 Key Questions

1. **"What if my quality gate is too strict?"** → You'll never deploy. Start with lenient thresholds and tighten over time.
2. **"What if it's too lenient?"** → Bad models reach production. Monitor your production metrics closely (Notebook 08).
3. **"How often should the pipeline run?"** → Depends on data freshness needs. Daily is common; some run hourly.

**Next →** `34_testing_ai_applications.ipynb` — Testing AI Applications: From Unit Tests to Behavioral Validation
